In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import io
import pickle as _pickle
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from itertools import product

import modal

from src.backtest import backtest

# Requires: modal deploy ../modal_train.py  (run once from the project root)
train_transformer_fn       = modal.Function.from_name('regime-trading-model', 'train_transformer')
train_transformer_fixed_fn = modal.Function.from_name('regime-trading-model', 'train_transformer_fixed')

def _decode_attr(value):
    if isinstance(value, dict) and value.get('__attr_type__') == 'dataframe_parquet':
        return pd.read_parquet(io.BytesIO(value['data']))
    return value

def _load_preds(raw: bytes) -> pd.DataFrame:
    """Deserialize Modal result -> DataFrame with .attrs restored."""
    result = _pickle.loads(raw)
    preds = pd.read_parquet(io.BytesIO(result['preds']))
    for k, v in result.get('attrs', {}).items():
        preds.attrs[k] = _decode_attr(v)
    return preds

def _pred_year_label(preds: pd.DataFrame, date_col: str = 'msgStamp') -> str:
    years = sorted(pd.to_datetime(preds[date_col]).dt.year.dropna().unique())
    return f'{years[0]}–{years[-1]}' if years else 'unknown'

def _pred_date_range_label(preds: pd.DataFrame, date_col: str = 'msgStamp') -> str:
    dates = pd.to_datetime(preds[date_col]).dropna()
    if dates.empty:
        return 'unknown'
    start, end = dates.min().date(), dates.max().date()
    return f'{start}' if start == end else f'{start} to {end}'

def _ensure_returns(preds: pd.DataFrame) -> pd.DataFrame:
    if 'ret_fopen' in preds.columns:
        return preds.copy()
    return preds.merge(df[['msgStamp', 'ret_fopen']], on='msgStamp', how='left')

def _average_seed_predictions(preds_by_seed: dict[int, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    pred_cols = []
    for seed, preds in preds_by_seed.items():
        col = f'pred_s{seed}'
        base = _ensure_returns(preds)[['msgStamp', 'trade_date', 'ret_fopen', 'pred']].copy()
        frames.append(base.rename(columns={'pred': col}))
        pred_cols.append(col)

    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(
            frame,
            on=['msgStamp', 'trade_date', 'ret_fopen'],
            how='inner',
            validate='one_to_one',
        )
    merged['pred'] = merged[pred_cols].mean(axis=1)
    return merged[['msgStamp', 'trade_date', 'ret_fopen', 'pred']]

print('Modal ready.')


In [ ]:
df = pd.read_parquet('../data/model_data.parquet')

x_cols = [c for c in df.columns if c.startswith('x')]
cond_cols = [c for c in df.columns if c.startswith('con')]

df['cond2'] = df['cond2'].ffill().bfill()
df['cond3'] = df['cond3'].ffill().bfill()


print(f'Features: {len(x_cols)}, Cond cols: {len(cond_cols)}, Rows: {len(df):,}')

## Objective and Architecture: Directed-Regime Attention Transformer

The objective is to design a transformer with a structural prior that treats **feature variables** and **regime variables** separately. The modeling assumption is: feature variables carry the direct return-prediction signal, while regime variables change how those feature signals should interact.

Two alternatives were tested before settling on this default:

- `typed_joint_attention`: feature and regime tokens use type-specific Q/K/V projections, then all tokens attend jointly with relation-type biases. This was too permissive and performed poorly in held-out validation.
- `regime_bias_attention`: regime variables produce feature-attention biases and feature gates, while feature tokens attend only to features. This was directionally reasonable but weaker than explicit regime-to-feature cross-attention.

The selected default is `directed_regime_attention`: separate feature and regime streams, followed by one-way regime-to-feature cross-attention. The final prediction head reads only feature tokens.

```
X: (B, 8)  C: (B, 3)   ← current step only (X_raw[:, -1, :] if window given)

Feature tokens:   H_f[j] = Linear_j(x_j)          j = 1..8   → (B, 8, 32)
Regime tokens:    H_r[k] = Linear_k(c_k)           k = 1..3   → (B, 3, 32)

── Directed-Regime Block × 2 ─────────────────────────────────────────────
Feature self-attention:
  H_f += MHA_f(LayerNorm(H_f), LayerNorm(H_f), LayerNorm(H_f))

Regime self-attention:
  H_r += MHA_r(LayerNorm(H_r), LayerNorm(H_r), LayerNorm(H_r))

Regime → feature cross-attention:
  query = LayerNorm(H_f)
  key   = LayerNorm(H_r)
  value = LayerNorm(H_r)
  H_f += MHA_cross(query=H_f, key=H_r, value=H_r)

Type-specific FFN:
  H_f += FFN_f(LayerNorm(H_f))       FFN: 32→128→GELU→32
  H_r += FFN_r(LayerNorm(H_r))       FFN: 32→128→GELU→32

── Readout ────────────────────────────────────────────────────────────────
z = flatten(H_f)                          (B, 8×32 = 256)
LayerNorm(256) → Linear(256→1) → squeeze → (B,)
```

### Default hyperparameters

| Hyperparameter | Default | Notes |
|---|---|---|
| `model_type` | `directed_regime_attention` | separate feature/regime streams + one-way regime→feature attention |
| `d_model` | 32 | per-token embedding dimension |
| `n_heads` | 4 | attention heads |
| `n_layers` | 2 | directed-regime blocks |
| `ffn_dim` | 128 | FFN hidden dim |
| `dropout` | 0.3 | applied after attention and FFN |
| `lr` | 1e-4 | AdamW learning rate |
| `weight_decay` | 0.1 | AdamW L2 (decoupled) |
| `grad_clip` | 1.0 | gradient norm clipping |
| `epochs` | 100 | max epochs |
| `batch_size` | 1024 | mini-batch size |
| `patience` | 6 | early stopping on inner validation Sharpe |
| `lr_scheduler_patience` | 3 | ReduceLROnPlateau patience on inner validation Sharpe |
| `warmup_fraction` | 0.01 | linear warmup, capped at one epoch |
| `x_clip` | 3.0 | feature clipping |
| `y_clip_quantile` | 0.01 | target winsorize (1st/99th pct) |

Each experiment is evaluated as a 3-seed equal-weight ensemble to reduce random-initialization noise.
LR scheduler: linear warmup over `warmup_fraction=0.01` of total steps, capped at one epoch → `ReduceLROnPlateau(mode='max', patience=3, factor=0.5, min_lr=1e-6)` on inner validation Sharpe.
Inner val: last 15% of training windows (temporal) · checkpoint selected by inner validation Sharpe · ~61k parameters per seed.


## 1. Validation run (train ≤ 2021, validate 2022-01-01 to 2023-06-30)

Trains once on all data through end of 2021 and predicts the validation period from 2022-01-01 to 2023-06-30.
The inner validation slice comes from the end of the training period, so early stopping is selected without touching the real validation period.
The default reported result is the equal-weight average of three random initializations.


In [ ]:
# Self-bootstrap so this default validation run works after a kernel restart.
import sys, pathlib, io
import pickle as _pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

_PROJECT_ROOT = pathlib.Path('..').resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import modal
from src.backtest import backtest

train_transformer_fixed_fn = modal.Function.from_name('regime-trading-model', 'train_transformer_fixed')

if 'df' not in globals():
    df = pd.read_parquet('../data/model_data.parquet')
    x_cols = [c for c in df.columns if c.startswith('x')]
    cond_cols = [c for c in df.columns if c.startswith('con')]
    df['cond2'] = df['cond2'].ffill().bfill()
    df['cond3'] = df['cond3'].ffill().bfill()

def _decode_attr(value):
    if isinstance(value, dict) and value.get('__attr_type__') == 'dataframe_parquet':
        return pd.read_parquet(io.BytesIO(value['data']))
    return value

def _load_preds(raw: bytes) -> pd.DataFrame:
    """Deserialize Modal result -> DataFrame with .attrs restored."""
    result = _pickle.loads(raw)
    preds = pd.read_parquet(io.BytesIO(result['preds']))
    for k, v in result.get('attrs', {}).items():
        preds.attrs[k] = _decode_attr(v)
    return preds

def _pred_date_range_label(preds: pd.DataFrame, date_col: str = 'msgStamp') -> str:
    dates = pd.to_datetime(preds[date_col]).dropna()
    if dates.empty:
        return 'unknown'
    start, end = dates.min().date(), dates.max().date()
    return f'{start}' if start == end else f'{start} to {end}'

def _ensure_returns(preds: pd.DataFrame) -> pd.DataFrame:
    if 'ret_fopen' in preds.columns:
        return preds.copy()
    return preds.merge(df[['msgStamp', 'ret_fopen']], on='msgStamp', how='left')

def _slice_date_range(preds: pd.DataFrame, start=None, end=None, date_col: str = 'msgStamp') -> pd.DataFrame:
    out = preds.copy()
    dates = pd.to_datetime(out[date_col])
    mask = pd.Series(True, index=out.index)
    if start is not None:
        mask &= dates >= pd.Timestamp(start)
    if end is not None:
        mask &= dates <= pd.Timestamp(end)
    return out.loc[mask].copy()

def _average_seed_predictions(preds_by_seed: dict[int, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    pred_cols = []
    for seed, preds in preds_by_seed.items():
        col = f'pred_s{seed}'
        base = _ensure_returns(preds)[['msgStamp', 'trade_date', 'ret_fopen', 'pred']].copy()
        frames.append(base.rename(columns={'pred': col}))
        pred_cols.append(col)

    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(
            frame,
            on=['msgStamp', 'trade_date', 'ret_fopen'],
            how='inner',
            validate='one_to_one',
        )
    merged['pred'] = merged[pred_cols].mean(axis=1)
    return merged[['msgStamp', 'trade_date', 'ret_fopen', 'pred']]

DEFAULT_MODEL_TYPE = 'directed_regime_attention'
DEFAULT_KWARGS = {
    'd_model': 32, 'n_heads': 4, 'n_layers': 2,
    'ffn_dim': 128, 'dropout': 0.2,
}
DEFAULT_LR = 1e-4
DEFAULT_WEIGHT_DECAY = 0.01
DEFAULT_GRAD_CLIP = 1.0
DEFAULT_PATIENCE = 6
DEFAULT_LR_SCHEDULER_PATIENCE = 3
DEFAULT_WARMUP_FRACTION = 0.01
DEFAULT_EPOCHS = 100
DEFAULT_BATCH_SIZE = 1024
DEFAULT_TRAIN_END_YEAR = 2021
DEFAULT_VAL_YEARS = [2022, 2023]
DEFAULT_VAL_START = '2022-01-01'
DEFAULT_VAL_END = '2023-06-30'
DEFAULT_TEST_START = '2023-07-01'
DEFAULT_TEST_END = None
DEFAULT_SEEDS = [7, 99, 2026]
DEFAULT_REFERENCE_SEED = DEFAULT_SEEDS[0]

print(
    f'Running validation fixed-split 3-seed average '
    f'(train ≤ {DEFAULT_TRAIN_END_YEAR}, validation {DEFAULT_VAL_START} to {DEFAULT_VAL_END})...'
)

handles = {
    seed: train_transformer_fixed_fn.spawn(
        model_type=DEFAULT_MODEL_TYPE,
        model_kwargs=DEFAULT_KWARGS,
        train_end_year=DEFAULT_TRAIN_END_YEAR,
        val_years=DEFAULT_VAL_YEARS,
        val_start=DEFAULT_VAL_START,
        val_end=DEFAULT_VAL_END,
        lr=DEFAULT_LR,
        weight_decay=DEFAULT_WEIGHT_DECAY,
        grad_clip=DEFAULT_GRAD_CLIP,
        patience=DEFAULT_PATIENCE,
        lr_scheduler_patience=DEFAULT_LR_SCHEDULER_PATIENCE,
        warmup_fraction=DEFAULT_WARMUP_FRACTION,
        epochs=DEFAULT_EPOCHS,
        batch_size=DEFAULT_BATCH_SIZE,
        random_state=seed,
    )
    for seed in DEFAULT_SEEDS
}



In [ ]:

preds_by_seed = {seed: _load_preds(handle.get()) for seed, handle in handles.items()}
preds_default_seed = preds_by_seed[DEFAULT_REFERENCE_SEED]
preds_default = _average_seed_predictions(preds_by_seed)
print('Validation run done.')

In [ ]:
_required = ['preds_default', '_ensure_returns']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the default validation model cell first; missing: {_missing}")

def _run_metrics(label, preds):
    dd = _ensure_returns(preds)
    bt = backtest(dd, feature_col='pred', ret_col='ret_fopen')
    return {
        'period': label,
        'sharpe': bt['sharpe_ratio'],
        'average_return': bt['average_return'],
        'n_rows': len(dd),
        'start': pd.to_datetime(dd['msgStamp']).min().date(),
        'end': pd.to_datetime(dd['msgStamp']).max().date(),
    }

runs_df = pd.DataFrame([_run_metrics('validation', preds_default)])
display(runs_df)
row = runs_df.iloc[0]
print(
    f'Default 3-seed average validation Sharpe: {row["sharpe"]:.4f}  '
    f'Return: {row["average_return"]:.6f}  ({row["start"]} to {row["end"]})'
)


In [ ]:
_required = ['preds_by_seed']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the default model cell first; missing: {_missing}")

def _unpack_gn(gn):
    """Handle both old (list of float) and new (list of dict) grad norm formats."""
    if not gn:
        return [], [], []
    if isinstance(gn[0], dict):
        return [g['mean'] for g in gn], [g['p95'] for g in gn], [g['max'] for g in gn]
    return gn, [], []

def _plot_default_seed_diagnostics(seed, history_preds):
    tl = history_preds.attrs.get('train_losses', [])
    vl = history_preds.attrs.get('val_losses', [])
    ts = history_preds.attrs.get('train_sharpes', [])
    vs = history_preds.attrs.get('val_sharpes', [])
    gn_mean, gn_p95, gn_max = _unpack_gn(history_preds.attrs.get('grad_norms', []))
    clip_line = globals().get('DEFAULT_GRAD_CLIP', 1.0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(
        f'Default Run Diagnostics (seed {seed}, {len(tl)} epochs, stop metric = inner-val Sharpe)',
        fontsize=13,
    )

    best_epoch_idx = None
    best_epoch = None
    if vs:
        vs_arr = np.asarray(vs, dtype=float)
        if np.isfinite(vs_arr).any():
            best_epoch_idx = int(np.nanargmax(vs_arr))
            best_epoch = best_epoch_idx + 1

    if tl:
        axes[0].plot(tl, label='train MSE')
        axes[0].plot(vl, label='inner val MSE')
        if best_epoch_idx is not None:
            axes[0].axvline(
                best_epoch_idx,
                color='gray',
                lw=1,
                linestyle='--',
                label=f'best Sharpe epoch {best_epoch}',
            )
        axes[0].set_title('MSE')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE')
        axes[0].set_ylim(0, 0.0015)
        axes[0].legend()
    else:
        axes[0].set_axis_off()

    if ts and vs:
        axes[1].plot(ts, label='train Sharpe')
        axes[1].plot(vs, label='inner val Sharpe')
        if best_epoch_idx is not None:
            axes[1].axvline(best_epoch_idx, color='gray', lw=1, linestyle='--', label=f'best epoch {best_epoch}')
        axes[1].axhline(0, color='black', lw=1, alpha=0.4)
        axes[1].set_title('Sharpe')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Daily Sharpe')
        axes[1].legend()
    else:
        axes[1].set_axis_off()

    if gn_mean:
        axes[2].plot(gn_mean, label='mean', color='tab:orange')
        if gn_p95:
            axes[2].plot(gn_p95, label='p95', color='tab:orange', linestyle='--', alpha=0.7)
        if gn_max:
            axes[2].plot(gn_max, label='max', color='tab:orange', linestyle=':', alpha=0.5)
        axes[2].axhline(clip_line, color='gray', lw=1, linestyle='--', label=f'clip={clip_line:g}')
        axes[2].set_ylim(0, 0.2)
        axes[2].set_title('Gradient Norm (pre-clip, per-epoch)')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('L2 norm')
        axes[2].legend()
    else:
        axes[2].set_axis_off()

    if tl:
        if best_epoch_idx is not None:
            print(
                f'seed {seed}: epochs={len(tl)}, best Sharpe epoch={best_epoch}, '
                f'best val Sharpe={vs[best_epoch_idx]:.4f}, '
                f'train Sharpe @ best={ts[best_epoch_idx]:.4f}, '
                f'train MSE @ best={tl[best_epoch_idx]:.6f}, '
                f'val MSE @ best={vl[best_epoch_idx]:.6f}'
            )
        else:
            print(f'seed {seed}: epochs={len(tl)}, no finite validation Sharpe found, best val MSE={min(vl):.6f}')

    plt.tight_layout()
    plt.show()

for seed, history_preds in sorted(preds_by_seed.items()):
    _plot_default_seed_diagnostics(seed, history_preds)


In [ ]:
_required = ['preds_by_seed', 'preds_default', '_ensure_returns']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the default validation model cell first; missing: {_missing}")

inner_val_preds = preds_default_seed.attrs.get('inner_val_preds')

if inner_val_preds is None:
    print('No inner validation predictions found. Redeploy ../modal_train.py and rerun the default job.')
else:
    inner_dd = _ensure_returns(inner_val_preds)
    inner_bt = backtest(inner_dd, feature_col='pred', ret_col='ret_fopen')
    inner_label = _pred_date_range_label(inner_dd)
    print(f'Default reference-seed inner-val Sharpe: {inner_bt["sharpe_ratio"]:.4f}  '
          f'Return: {inner_bt["average_return"]:.6f}  ({inner_label})')

    ax = inner_bt['cumulative_returns'].plot(
        figsize=(10, 4),
        title=f'Default Run — Reference Seed Inner Validation PnL ({inner_label})  |  Sharpe={inner_bt["sharpe_ratio"]:.3f}',
    )
    ax.axhline(0, color='black', lw=1, alpha=0.5)
    ax.set_ylabel('Cumulative PnL')
    plt.tight_layout()
    plt.show()

validation_dd = _ensure_returns(preds_default)
validation_bt = backtest(validation_dd, feature_col='pred', ret_col='ret_fopen')
validation_label = _pred_date_range_label(validation_dd)
print(f'Default 3-seed validation Sharpe: {validation_bt["sharpe_ratio"]:.4f}  '
      f'Return: {validation_bt["average_return"]:.6f}  ({validation_label})')

ax = validation_bt['cumulative_returns'].plot(
    figsize=(10, 4),
    title=f'Default Run — 3-Seed Validation PnL ({validation_label})  |  Sharpe={validation_bt["sharpe_ratio"]:.3f}',
)
ax.axhline(0, color='black', lw=1, alpha=0.5)
ax.set_ylabel('Cumulative PnL')
plt.tight_layout()
plt.show()


In [ ]:
_required = ['preds_default', 'runs_df', '_ensure_returns']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the default validation model cell first; missing: {_missing}")

NOTEBOOK_RESULTS_DIR = pathlib.Path('../experiments/results/notebook_default')
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_ensure_returns(preds_default).to_parquet(
    NOTEBOOK_RESULTS_DIR / f'{DEFAULT_MODEL_TYPE}_3seed_average_validation_2022_2023H1.parquet',
    index=False,
)
runs_df.to_csv(NOTEBOOK_RESULTS_DIR / f'{DEFAULT_MODEL_TYPE}_3seed_average_validation_summary.csv', index=False)
print(f'Saved averaged validation notebook results to {NOTEBOOK_RESULTS_DIR.resolve()}')


## 2. Test run (train through validation end, predict from 2023-07-01)

After checking validation diagnostics, train the same model through the end of the validation period (`2023-06-30`) and predict the test period from `2023-07-01` onward.
This is a separate final-evaluation run: validation rows are now allowed in training, and test rows are not.


In [ ]:
_required = ['train_transformer_fixed_fn', '_load_preds', '_average_seed_predictions']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the default validation model cell first; missing: {_missing}")

FINAL_TRAIN_END_DATE = DEFAULT_VAL_END
FINAL_TRAIN_END_YEAR = int(FINAL_TRAIN_END_DATE[:4])

if 'preds_test' in globals() and 'preds_test_by_seed' in globals():
    print('Test predictions are already loaded in this kernel.')
elif 'test_handles' in globals():
    print('Collecting existing final test Modal jobs...')
    preds_test_by_seed = {seed: _load_preds(handle.get()) for seed, handle in test_handles.items()}
    preds_test_seed = preds_test_by_seed[DEFAULT_REFERENCE_SEED]
    preds_test = _average_seed_predictions(preds_test_by_seed)
    print('Test run done.')
else:
    print(
        f'Running final test 3-seed average '
        f'(train through {FINAL_TRAIN_END_DATE}, test from {DEFAULT_TEST_START})...'
    )

    test_handles = {
        seed: train_transformer_fixed_fn.spawn(
            model_type=DEFAULT_MODEL_TYPE,
            model_kwargs=DEFAULT_KWARGS,
            train_end_year=FINAL_TRAIN_END_YEAR,
            train_end_date=FINAL_TRAIN_END_DATE,
            val_years=[FINAL_TRAIN_END_YEAR],
            val_start=DEFAULT_TEST_START,
            val_end=DEFAULT_TEST_END,
            lr=DEFAULT_LR,
            weight_decay=DEFAULT_WEIGHT_DECAY,
            grad_clip=DEFAULT_GRAD_CLIP,
            patience=DEFAULT_PATIENCE,
            lr_scheduler_patience=DEFAULT_LR_SCHEDULER_PATIENCE,
            warmup_fraction=DEFAULT_WARMUP_FRACTION,
            epochs=DEFAULT_EPOCHS,
            batch_size=DEFAULT_BATCH_SIZE,
            random_state=seed,
        )
        for seed in DEFAULT_SEEDS
    }

    preds_test_by_seed = {seed: _load_preds(handle.get()) for seed, handle in test_handles.items()}
    preds_test_seed = preds_test_by_seed[DEFAULT_REFERENCE_SEED]
    preds_test = _average_seed_predictions(preds_test_by_seed)
    print('Test run done.')


In [ ]:
# c2: diagnostics for the final test-period model run.
_required = ['preds_test_by_seed']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the final test model cell first; missing: {_missing}")

def _unpack_gn(gn):
    """Handle both old (list of float) and new (list of dict) grad norm formats."""
    if not gn:
        return [], [], []
    if isinstance(gn[0], dict):
        return [g['mean'] for g in gn], [g['p95'] for g in gn], [g['max'] for g in gn]
    return gn, [], []

def _plot_test_seed_diagnostics(seed, history_preds):
    tl = history_preds.attrs.get('train_losses', [])
    vl = history_preds.attrs.get('val_losses', [])
    ts = history_preds.attrs.get('train_sharpes', [])
    vs = history_preds.attrs.get('val_sharpes', [])
    gn_mean, gn_p95, gn_max = _unpack_gn(history_preds.attrs.get('grad_norms', []))
    clip_line = globals().get('DEFAULT_GRAD_CLIP', 1.0)
    train_end = history_preds.attrs.get('train_end_date', globals().get('FINAL_TRAIN_END_DATE', 'validation end'))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(
        f'Final Test-Run Diagnostics (seed {seed}, train through {train_end}, {len(tl)} epochs)',
        fontsize=13,
    )

    best_epoch_idx = None
    best_epoch = None
    if vs:
        vs_arr = np.asarray(vs, dtype=float)
        if np.isfinite(vs_arr).any():
            best_epoch_idx = int(np.nanargmax(vs_arr))
            best_epoch = best_epoch_idx + 1

    if tl:
        axes[0].plot(tl, label='train MSE')
        axes[0].plot(vl, label='inner val MSE')
        if best_epoch_idx is not None:
            axes[0].axvline(
                best_epoch_idx,
                color='gray',
                lw=1,
                linestyle='--',
                label=f'best Sharpe epoch {best_epoch}',
            )
        axes[0].set_title('MSE')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE')
        axes[0].set_ylim(0, 0.0015)
        axes[0].legend()
    else:
        axes[0].set_axis_off()

    if ts and vs:
        axes[1].plot(ts, label='train Sharpe')
        axes[1].plot(vs, label='inner val Sharpe')
        if best_epoch_idx is not None:
            axes[1].axvline(best_epoch_idx, color='gray', lw=1, linestyle='--', label=f'best epoch {best_epoch}')
        axes[1].axhline(0, color='black', lw=1, alpha=0.4)
        axes[1].set_title('Sharpe')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Daily Sharpe')
        axes[1].legend()
    else:
        axes[1].set_axis_off()

    if gn_mean:
        axes[2].plot(gn_mean, label='mean', color='tab:orange')
        if gn_p95:
            axes[2].plot(gn_p95, label='p95', color='tab:orange', linestyle='--', alpha=0.7)
        if gn_max:
            axes[2].plot(gn_max, label='max', color='tab:orange', linestyle=':', alpha=0.5)
        axes[2].axhline(clip_line, color='gray', lw=1, linestyle='--', label=f'clip={clip_line:g}')
        axes[2].set_ylim(0, 0.2)
        axes[2].set_title('Gradient Norm (pre-clip, per-epoch)')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('L2 norm')
        axes[2].legend()
    else:
        axes[2].set_axis_off()

    if tl:
        if best_epoch_idx is not None:
            print(
                f'seed {seed}: epochs={len(tl)}, best Sharpe epoch={best_epoch}, '
                f'best inner-val Sharpe={vs[best_epoch_idx]:.4f}, '
                f'train Sharpe @ best={ts[best_epoch_idx]:.4f}, '
                f'train MSE @ best={tl[best_epoch_idx]:.6f}, '
                f'inner-val MSE @ best={vl[best_epoch_idx]:.6f}'
            )
        else:
            print(f'seed {seed}: epochs={len(tl)}, no finite inner-val Sharpe found, best inner-val MSE={min(vl):.6f}')

    plt.tight_layout()
    plt.show()

for seed, history_preds in sorted(preds_test_by_seed.items()):
    _plot_test_seed_diagnostics(seed, history_preds)


In [ ]:
_required = ['preds_test', '_ensure_returns']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the final test model cell first; missing: {_missing}")

test_dd = _ensure_returns(preds_test)
if len(test_dd) == 0:
    test_runs_df = pd.DataFrame([{
        'period': 'test',
        'sharpe': np.nan,
        'average_return': np.nan,
        'n_rows': 0,
        'start': None,
        'end': None,
    }])
    display(test_runs_df)
    print('No test rows are available after 2023-07-01 in the data returned by Modal.')
else:
    test_bt = backtest(test_dd, feature_col='pred', ret_col='ret_fopen')
    test_label = _pred_date_range_label(test_dd)
    test_runs_df = pd.DataFrame([{
        'period': 'test',
        'sharpe': test_bt['sharpe_ratio'],
        'average_return': test_bt['average_return'],
        'n_rows': len(test_dd),
        'start': pd.to_datetime(test_dd['msgStamp']).min().date(),
        'end': pd.to_datetime(test_dd['msgStamp']).max().date(),
    }])
    display(test_runs_df)
    print(f'Final 3-seed test Sharpe: {test_bt["sharpe_ratio"]:.4f}  '
          f'Return: {test_bt["average_return"]:.6f}  ({test_label})')

    ax = test_bt['cumulative_returns'].plot(
        figsize=(10, 4),
        title=f'Final Run — 3-Seed Test PnL ({test_label})  |  Sharpe={test_bt["sharpe_ratio"]:.3f}',
    )
    ax.axhline(0, color='black', lw=1, alpha=0.5)
    ax.set_ylabel('Cumulative PnL')
    plt.tight_layout()
    plt.show()


In [ ]:
_required = ['preds_test', 'test_runs_df', '_ensure_returns']
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the final test result cell first; missing: {_missing}")

NOTEBOOK_RESULTS_DIR = pathlib.Path('../experiments/results/notebook_default')
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_ensure_returns(preds_test).to_parquet(
    NOTEBOOK_RESULTS_DIR / f'{DEFAULT_MODEL_TYPE}_3seed_average_test_from_2023H2.parquet',
    index=False,
)
test_runs_df.to_csv(NOTEBOOK_RESULTS_DIR / f'{DEFAULT_MODEL_TYPE}_3seed_average_test_summary.csv', index=False)
print(f'Saved averaged test notebook results to {NOTEBOOK_RESULTS_DIR.resolve()}')
